# New Backgrounds - Sanity Checks and Verifications

First, we load all of the background and signal files.

In [ ]:
from pathlib import Path
from diquark.config.config_manager import ConfigManager
from diquark.data.loader import DataLoader

config_files_dir = Path(".").absolute().parent / "diquark" / "config"

config_file_path = config_files_dir / "New_Features" / "New_Backgrounds" / "ATLAS_136_S8000_B7500.yaml"
assert config_file_path.exists()

config = ConfigManager(config_file_path)

# Load backgrounds config file
backgrounds_config_file = config.get_required("data.backgrounds_file")
backgrounds_config_file_path = (
    config_files_dir / "Backgrounds" / backgrounds_config_file
)
backgrounds_config = ConfigManager(backgrounds_config_file_path)

# Sanity check for matching phase space cuts
assert config.get_required(
    "data.mass_cut"
) == backgrounds_config.get_required("backgrounds.mass_cut")

backgrounds_directory = Path(
    backgrounds_config.get_required("backgrounds.base_directory")
)
backgrounds_file_names = backgrounds_config.get("backgrounds.file_names", {})
backgrounds_file_name_mapping = backgrounds_config.get(
    "backgrounds.file_name_mapping", {}
)

files = set()

def check_root_file_exists(path: str):
    path = Path(path)
    if not path.exists():
        raise Exception(f"Could not find background file at path '{path}'")
    files.add(path)
    return path

background_path_dict = {
    f"BKG:{key}": check_root_file_exists(path)
    for key, path in backgrounds_file_names.items()
}
assert len(files) == len(background_path_dict.keys())

def find_root_file_by_name(filename: str) -> Path:
    results = backgrounds_directory.glob(f"*{filename}*.root")
    for path in results:
        files.add(path)
        return path
    else:
        raise Exception(
            f"Could not find background file with filename '{filename}'"
        )

background_path_dict |= {
    f"BKG:{key}": find_root_file_by_name(filename)
    for key, filename in backgrounds_file_name_mapping.items()
}
assert len(files) == len(background_path_dict.keys())

# Load signal config file
signal_config_file = config.get_required("data.signal_file")
backgrounds_config_file_path = config_files_dir / "Signals" / signal_config_file
signal_config = ConfigManager(backgrounds_config_file_path)

signal_file = Path(signal_config.get("signal.file"))
path_dict = background_path_dict | {"SIG:Suu": signal_file}

data_loader = DataLoader(
    path_dict,
    index_start=0,
    index_stop=None,
)

In [ ]:
mass_cut = config.get_required("data.mass_cut")
data = data_loader.load_data(mass_cut)

In [ ]:
from tqdm.contrib.concurrent import thread_map
from diquark.features.feature_extractor import FeatureExtractor

print("Extracting features...")
feature_extractor = FeatureExtractor(n_jets=32, chi_mass=2000, suu_mass=8000)

features = thread_map(
    feature_extractor.compute_all,
    data.values(),
    max_workers=32,
    desc="Extracting features",
)

print(f"Working with {len(features[0].keys())} feature columns")

assert len(features[0].keys()) == len(feature_extractor.feature_names), (
    f"Number of extracted features ({len(features[0].keys())}) doesn't match number of feature names defined on feature extractor object ({len(feature_extractor.feature_names)})"
)

datasets = dict(zip(data.keys(), features))

In [ ]:
first_event = data['SIG:Suu'][0]
first_event

In [ ]:
p_T = first_event["Jet/Jet.PT"]
eta = first_event["Jet/Jet.Eta"]
phi = first_event["Jet/Jet.Phi"]

# Compute components in Cartesian coordinates
p_x = p_T * np.cos(phi)
p_y = p_T * np.sin(phi)
p_z = p_T * np.sinh(eta)
print("p_x =", p_x)
print("p_y =", p_y)
print("p_z =", p_z)

# Compute jet energy
energy = p_T * np.cosh(eta)
print("E =", energy)

In [ ]:
p_x_total = np.sum(p_x)
p_y_total = np.sum(p_y)
p_z_total = np.sum(p_z)

total_energy = np.sum(energy)

total_mass_squared = (
    # |sum energies|^2 - ||sum momentum vectors||^2
    total_energy**2 - p_x_total**2 - p_y_total**2 - p_z_total**2
)

mass = np.sqrt(total_mass_squared)
print("m total =", mass)

In [ ]:
features['combined_invariant_mass'][0]

In [ ]:
import numpy as np

percentages = {}
for key, features in datasets.items():
    percentages[key] = np.sum(features['combined_invariant_mass'] <= 7500) / len(features['combined_invariant_mass'])

In [ ]:
percentages

In [ ]:
import pandas as pd

In [ ]:
results = pd.DataFrame.from_records([percentages]).transpose()
results.columns = ["Percentage events combined invariant mass <= 7500"]
results